# 08 Fairness audit — C-02 thresholds and C-03 group errors (validation year 2023)

Written by Yiling at merge (2026-11-15) because C-02 was due before the freeze; Yixuan takes over C-03/C-04 on the test year.

- **Threshold rule (frozen):** for each model × feature set, predict 1 when `y_prob >= t`, with `t` chosen so that the 2023 predicted share of 1s equals the actual 2023 share (`fairness.choose_threshold`; ties go to the neighbouring score closer to that share). Each specification gets its own `t`.
- **Errors are relative to the historical decision label** (bail set or remand), not to a "correct" decision.
- Groups with fewer than `MIN_GROUP_N` cases are not reported; FPR/FNR are blank when the group has fewer than `MIN_GROUP_N` actual 0s/1s.
- 2024 is not used here. The same thresholds carry over to BD-07 unchanged.

In [1]:
import sys; sys.path.append("..")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src import config
from src.evaluate import load_predictions
from src.fairness import choose_threshold, group_metrics, predict_label, residuals

assert config.ROOT == Path.cwd().resolve().parent, "Run from the notebooks directory."
SPECS = [("xgb", "CASE"), ("xgb", "CASE_COURT"), ("xgb", "CASE_COURT_JUDGE"),   # final candidate family
         ("logit", "CASE"), ("logit", "CASE_COURT"), ("logit", "CASE_COURT_JUDGE")]  # simple reference
GROUPS = list(config.FAIRNESS_GROUPS)                                            # race, ethnicity, gender, age_group

In [2]:
pred = load_predictions()
pred = pred.loc[pred["split"].eq("valid") & pred["target"].eq(config.TARGET_Q1)]
assert not pred[config.YEAR_COL].eq(config.TEST_YEAR).any()
meta_cols = [config.ID_COL, config.YEAR_COL, config.SPLIT_COL, config.MAIN_SAMPLE_COL,
             "county_name", "judge_id", *GROUPS]
meta = pd.read_parquet(config.ANALYTIC_FILE, columns=meta_cols,
                       filters=[(config.YEAR_COL, "==", config.VALID_YEAR)])
meta = meta.loc[meta[config.MAIN_SAMPLE_COL]]
print("2023 main-sample cases:", len(meta))
for model, fs in SPECS:
    n = pred.loc[pred.model.eq(model) & pred.feature_set.eq(fs)].shape[0]
    assert n == len(meta), (model, fs, n)
print("Every specification has one 2023 prediction per case.")

2023 main-sample cases: 120457


Every specification has one 2023 prediction per case.


## C-02 Thresholds

In [3]:
rows = []
for model, fs in SPECS:
    part = pred.loc[pred.model.eq(model) & pred.feature_set.eq(fs)]
    t = choose_threshold(part["y_prob"], part["y_true"])
    yhat = predict_label(part["y_prob"], t)
    rows.append({"model": model, "feature_set": fs, "threshold": t, "rule": "y_prob >= threshold",
                 "n": len(part), "actual_share": part["y_true"].mean(), "predicted_share": yhat.mean(),
                 "fpr": ((yhat == 1) & (part["y_true"] == 0)).sum() / (part["y_true"] == 0).sum(),
                 "fnr": ((yhat == 0) & (part["y_true"] == 1)).sum() / (part["y_true"] == 1).sum()})
thresholds = pd.DataFrame(rows)
thresholds.to_csv(config.REPORTS / "c02_thresholds.csv", index=False)
thresholds.round(5)

,model,feature_set,threshold,rule,n,actual_share,predicted_share,fpr,fnr
0,xgb,CASE,0.35201,y_prob >= threshold,120457,0.12818,0.12820,0.05427,0.36898
1,xgb,CASE_COURT,0.34157,y_prob >= threshold,120457,0.12818,0.12818,0.05334,0.36282
2,xgb,CASE_COURT_JUDGE,0.35102,y_prob >= threshold,120457,0.12818,0.12818,0.05248,0.35693
3,logit,CASE,0.34142,y_prob >= threshold,120457,0.12818,0.12818,0.05568,0.37869
4,logit,CASE_COURT,0.34064,y_prob >= threshold,120457,0.12818,0.12817,0.05551,0.37766
5,logit,CASE_COURT_JUDGE,0.34313,y_prob >= threshold,120457,0.12818,0.12818,0.05436,0.36975


## C-03 Group errors and calibration (2023)

In [4]:
tables = []
for model, fs in SPECS:
    part = pred.loc[pred.model.eq(model) & pred.feature_set.eq(fs)]
    t = float(thresholds.set_index(["model", "feature_set"]).loc[(model, fs), "threshold"])
    for g in GROUPS:
        tables.append(group_metrics(part, meta, g, t))
group_table = pd.concat(tables, ignore_index=True)
group_table.to_csv(config.REPORTS / "c03_group_metrics_valid.csv", index=False)
show = ["model", "feature_set", "group", "n", "base_rate", "mean_pred", "calibration_gap", "pred_pos_rate", "fpr", "fnr", "roc_auc"]
group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb"), show].round(4)

,model,feature_set,group,n,base_rate,mean_pred,calibration_gap,pred_pos_rate,fpr,fnr,roc_auc
0,xgb,CASE,American Indian/Alaskan Native,407,0.0737,0.0521,-0.0216,0.0344,0.0186,NaN,NaN
1,xgb,CASE,Asian/Pacific Islander,7219,0.0907,0.0711,-0.0196,0.0690,0.0274,0.5145,0.9418
2,xgb,CASE,Black,68765,0.1431,0.1376,-0.0054,0.1529,0.0654,0.3233,0.9301
3,xgb,CASE,Unknown,1768,0.1431,0.1124,-0.0307,0.1227,0.0561,0.4783,0.9156
4,xgb,CASE,White,42289,0.1102,0.0966,-0.0136,0.0993,0.0418,0.4367,0.9286
17,xgb,CASE_COURT,American Indian/Alaskan Native,407,0.0737,0.0528,-0.0209,0.0344,0.0186,NaN,NaN
18,xgb,CASE_COURT,Asian/Pacific Islander,7219,0.0907,0.0697,-0.0210,0.0687,0.0262,0.5053,0.9421
19,xgb,CASE_COURT,Black,68765,0.1431,0.1366,-0.0065,0.1530,0.0644,0.3168,0.9311
20,xgb,CASE_COURT,Unknown,1768,0.1431,0.1121,-0.0310,0.1250,0.0554,0.4585,0.9155
21,xgb,CASE_COURT,White,42289,0.1102,0.0964,-0.0138,0.0990,0.0410,0.4324,0.9295


Does adding court and judge change the gaps between groups? (XGBoost, race; difference = CASE_COURT_JUDGE − CASE)

In [5]:
x = group_table.loc[group_table.model.eq("xgb")].set_index(["group_col", "group", "feature_set"])
gap = (x.xs("CASE_COURT_JUDGE", level="feature_set")[["calibration_gap", "pred_pos_rate", "fpr", "fnr"]]
       - x.xs("CASE", level="feature_set")[["calibration_gap", "pred_pos_rate", "fpr", "fnr"]])
gap.to_csv(config.REPORTS / "c03_judge_change_valid.csv")
gap.round(4)

calibration_gap  pred_pos_rate  \
group_col group                                                            
race      American Indian/Alaskan Native          -0.0005         0.0000   
          Asian/Pacific Islander                   0.0015         0.0030   
          Black                                    0.0002        -0.0011   
          Unknown                                  0.0033         0.0090   
          White                                    0.0009         0.0008   
ethnicity Hispanic                                 0.0003        -0.0008   
          Non Hispanic                             0.0006         0.0002   
          Unknown                                  0.0036         0.0057   
gender    Female                                  -0.0005        -0.0005   
          Male                                     0.0008         0.0001   
          Unknown                                  0.0003         0.0011   
age_group 18-24                                   -0.0004        -0.0039   
          25-34                                    0.0004         0.0011   
          35-44                                    0.0016         0.0006   
          45-54                                    0.0002        -0.0021   
          55+                                      0.0007         0.0031   
          <18                                     -0.0004         0.0060   

                                             fpr     fnr  
group_col group                                           
race      American Indian/Alaskan Native -0.0027     NaN  
          Asian/Pacific Islander         -0.0012 -0.0458  
          Black                          -0.0023 -0.0059  
          Unknown                         0.0026 -0.0474  
          White                          -0.0014 -0.0182  
ethnicity Hispanic                       -0.0025 -0.0119  
          Non Hispanic                   -0.0015 -0.0115  
          Unknown                         0.0006 -0.0340  
gender    Female                         -0.0016 -0.0201  
          Male                           -0.0018 -0.0111  
          Unknown                        -0.0031 -0.0368  
age_group 18-24                          -0.0046 -0.0003  
          25-34                          -0.0006 -0.0134  
          35-44                          -0.0008 -0.0099  
          45-54                          -0.0030 -0.0045  
          55+                            -0.0004 -0.0331  
          <18                            -0.0190 -0.0433

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
race = group_table.loc[group_table.group_col.eq("race") & group_table.model.eq("xgb")
                       & group_table.feature_set.isin(["CASE", "CASE_COURT_JUDGE"])]
order = race.loc[race.feature_set.eq("CASE")].sort_values("n", ascending=False)["group"].tolist()
for ax, metric, title in zip(axes, ["fpr", "fnr"], ["False positive rate", "False negative rate"]):
    for k, (fs, marker) in enumerate([("CASE", "o"), ("CASE_COURT_JUDGE", "s")]):
        r = race.loc[race.feature_set.eq(fs)].set_index("group").reindex(order)
        ax.scatter(r[metric], np.arange(len(order)) + (k - 0.5) * 0.25, marker=marker, label=fs)
    ax.set_title(title); ax.set_yticks(range(len(order))); ax.set_yticklabels(order); ax.grid(axis="x", alpha=0.3)
axes[0].legend(frameon=False, loc="lower right")
fig.suptitle("XGBoost, 2023 validation, policy threshold (relative to historical decisions)")
fig.tight_layout()
(config.REPORTS / "figures").mkdir(exist_ok=True)
fig.savefig(config.REPORTS / "figures" / "c03_race_errors_valid.png", dpi=200)
plt.close(fig)
print("Saved reports/figures/c03_race_errors_valid.png")

Saved reports/figures/c03_race_errors_valid.png


## C-04 preview: county residuals (XGBoost CASE, 2023)

Observed rate minus mean predicted probability. Positive = more bail/remand than the case-and-record model predicts. Not a causal effect. Judge residuals are left for C-04 on the test year.

In [7]:
part = pred.loc[pred.model.eq("xgb") & pred.feature_set.eq("CASE")]
county_res = residuals(part, meta, "county_name")
county_res.to_csv(config.REPORTS / "c04_county_residuals_valid.csv", index=False)
county_res.round(4)

,model,feature_set,by,group,n,observed_rate,mean_pred,residual,ci_low,ci_high
0,xgb,CASE,county_name,Bronx,20938,0.1162,0.1372,-0.0210,-0.0243,-0.0176
1,xgb,CASE,county_name,Kings,38662,0.1182,0.1198,-0.0017,-0.0042,0.0008
2,xgb,CASE,county_name,New York,25673,0.1473,0.1242,0.0231,0.0198,0.0265
3,xgb,CASE,county_name,Queens,28908,0.1376,0.1021,0.0355,0.0325,0.0385
4,xgb,CASE,county_name,Richmond,6276,0.1080,0.1020,0.0060,-0.0002,0.0115


## C-05 Disparity analysis in the Week 8 Class 2 form (XGBoost `CASE_COURT_JUDGE`, 2023)

Added 2026-11-19 after the instructor's feedback: a model whose errors differ by group cannot be used in practice, so we must **adjust**, not only report. Steps follow the lecture ("Disparity Analysis in Five Steps").

1. **The decision**: flag = `y_prob >= 0.351` (bail set or remand predicted). A false positive is a case the judge released that the model flags; that error falls on the defendant. A false negative is a bail/remand case the model misses.
2. **The groups**: race (audit variable only, never a feature). Cells with n < `MIN_GROUP_N` are not reported.
3–4. **Confusion matrix and four rates per group**, gap and ratio, four-fifths rule on the flag rate.
5. **Judge and act**: we treat FPR as the rate that matters (the harm of a false positive falls on the defendant) and show the after-the-model fix from class, a group-specific threshold, with its cost.

Errors are relative to the historical decision label. Precision and calibration are also reported because, as in the COMPAS case from class, equal precision and unequal FPR can hold at the same time when base rates differ.

In [8]:
from src.fairness import disparity_rates, rate_comparison, equalizing_threshold, missing_by_group
FINAL = ("xgb", "CASE_COURT_JUDGE")
T = float(thresholds.set_index(["model", "feature_set"]).loc[FINAL, "threshold"])
final_pred = pred.loc[pred.model.eq(FINAL[0]) & pred.feature_set.eq(FINAL[1])]

# Steps 3-4: confusion matrix and the four rates by race
rates_race = disparity_rates(final_pred, meta, "race", T)
rates_race.to_csv(config.REPORTS / "c05_rates_by_race_valid.csv", index=False)
display(rates_race.set_index("group")[["n", "share_pos", "TP", "FP", "FN", "TN", "flag_rate", "tpr", "fpr", "precision"]].round(3))

# Gap and ratio for the two largest groups; four-fifths rule on the flag rate
compare = rate_comparison(rates_race, "Black", "White")
compare.to_csv(config.REPORTS / "c05_black_white_comparison_valid.csv", index=False)
print("Black vs White (gap = Black − White; ratio = smaller / larger; 1 = equal)")
display(compare.round(3))
ff = compare.set_index("rate").loc["flag_rate", "ratio"]
print(f"Four-fifths rule on flag rate: ratio {ff:.2f} -> {'passes' if ff >= 0.80 else 'FAILS'}")

,n,share_pos,TP,FP,FN,TN,flag_rate,tpr,fpr,precision
group,,,,,,,,,,
American Indian/Alaskan Native,407,0.074,8,6,22,371,0.034,NaN,0.016,NaN
Asian/Pacific Islander,7219,0.091,348,172,307,6392,0.072,0.531,0.026,0.669
Black,68765,0.143,6717,3720,3123,55205,0.152,0.683,0.063,0.644
Unknown,1768,0.143,144,89,109,1426,0.132,0.569,0.059,0.618
White,42289,0.110,2710,1522,1950,36107,0.100,0.582,0.040,0.640


Black vs White (gap = Black − White; ratio = smaller / larger; 1 = equal)


,rate,Black,White,gap,ratio,four_fifths
0,flag_rate,0.152,0.100,0.052,0.659,False
1,tpr,0.683,0.582,0.101,0.852,NaN
2,fpr,0.063,0.040,0.023,0.641,NaN
3,precision,0.644,0.640,0.003,0.995,NaN


Four-fifths rule on flag rate: ratio 0.66 -> FAILS


In [9]:
# Grouped bar chart in the course style (SPEC 25 Rule 6)
rates = ["flag_rate", "tpr", "fpr", "precision"]
labels = ["Flag rate", "TPR", "FPR", "Precision"]
r = rates_race.set_index("group")
x = np.arange(len(rates))
fig, ax = plt.subplots(figsize=(7, 3.8))
for i, (grp, color) in enumerate([("Black", "#4E2A84"), ("White", "#D86018")]):
    vals = r.loc[grp, rates].astype(float)
    ax.bar(x + (i - 0.5) * 0.38, vals, 0.38, color=color, label=f"{grp} (n = {int(r.loc[grp, 'n']):,})")
    for xi, v in zip(x, vals):
        ax.text(xi + (i - 0.5) * 0.38, v + 0.015, f"{v:.2f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylim(0, 1.0); ax.legend(frameon=False)
ax.set_title("Same model, same threshold, two groups (2023, relative to historical decisions)", fontsize=10)
fig.tight_layout(); fig.savefig(config.REPORTS / "figures" / "c05_rates_by_race_valid.png", dpi=200); plt.close(fig)
print("Saved reports/figures/c05_rates_by_race_valid.png")

Saved reports/figures/c05_rates_by_race_valid.png


### Step 5: after-the-model adjustment, a group-specific threshold

Raise the threshold for the Black group until its FPR equals the White group's FPR at the common threshold (chosen on 2023 only; the test year reuses the value). We report what it costs. Group-specific thresholds can be legally restricted in real decisions (lecture, Part 26); this shows the trade-off, it is not a deployment recommendation.

In [10]:
fpr_white = float(rates_race.set_index("group").loc["White", "fpr"])
T_BLACK = equalizing_threshold(final_pred, meta, "race", "Black", target_fpr=fpr_white)
adjusted = disparity_rates(final_pred, meta, "race", T, thresholds_by_group={"Black": T_BLACK})

before = rates_race.set_index("group").loc[["Black", "White"], ["threshold", "flag_rate", "tpr", "fpr", "precision"]]
after = adjusted.set_index("group").loc[["Black", "White"], ["threshold", "flag_rate", "tpr", "fpr", "precision"]]
threshold_table = pd.concat({"common threshold": before, "Black threshold raised": after}, names=["scenario"])
threshold_table.to_csv(config.REPORTS / "c05_group_threshold_valid.csv")
print(f"Common threshold {T:.3f}; Black-group threshold for equal FPR: {T_BLACK:.3f}")
display(threshold_table.round(3))

# Overall cost of the adjustment (all 2023 cases)
d = final_pred.merge(meta[[config.ID_COL, "race"]], on=config.ID_COL)
def overall(flag):
    y = d["y_true"].eq(1)
    return {"flag_rate": flag.mean(), "tpr": (flag & y).sum() / y.sum(), "fpr": (flag & ~y).sum() / (~y).sum(),
            "precision": (flag & y).sum() / flag.sum()}
common = d["y_prob"].ge(T)
adj = np.where(d["race"].eq("Black"), d["y_prob"].ge(T_BLACK), common)
overall_cost = pd.DataFrame({"common": overall(common), "adjusted": overall(pd.Series(adj))}).T
overall_cost.to_csv(config.REPORTS / "c05_overall_cost_valid.csv")
display(overall_cost.round(3))

Common threshold 0.351; Black-group threshold for equal FPR: 0.488


threshold  flag_rate    tpr    fpr  precision
scenario               group                                               
common threshold       Black      0.351      0.152  0.683  0.063      0.644
                       White      0.351      0.100  0.582  0.040      0.640
Black threshold raised Black      0.488      0.118  0.580  0.040      0.706
                       White      0.351      0.100  0.582  0.040      0.640

,flag_rate,tpr,fpr,precision
common,0.128,0.643,0.052,0.643
adjusted,0.109,0.578,0.040,0.681


### Checklist step 3: do missing values differ by group?

`dcjs_missing` = 1 when the prior-record fields are absent (median-imputed in the pipeline, with the flag as a feature).

In [11]:
miss_cols = [config.ID_COL, config.YEAR_COL, config.MAIN_SAMPLE_COL, "dcjs_missing", "race"]
miss = pd.read_parquet(config.ANALYTIC_FILE, columns=miss_cols,
                       filters=[(config.YEAR_COL, "in", config.TRAIN_YEARS + [config.VALID_YEAR])])
miss = miss.loc[miss[config.MAIN_SAMPLE_COL]]
missing_race = missing_by_group(miss, "dcjs_missing", "race")
missing_race.to_csv(config.REPORTS / "c05_missing_by_race.csv", index=False)
missing_race.round(4)

,race,n,share_missing
0,American Indian/Alaskan Native,752,0.0093
1,Asian/Pacific Islander,13934,0.0127
2,Black,179835,0.0094
3,Unknown,8639,0.0155
4,White,110472,0.0108


In [12]:
assert not group_table.empty and (group_table["n"] >= config.MIN_GROUP_N).all()
assert abs(adjusted.set_index("group").loc["Black", "fpr"] - fpr_white) < 0.002, "adjusted Black FPR should match White FPR"
print("C-02/C-03/C-04 and C-05 validation audit written: reports/c02_thresholds.csv, c03_group_metrics_valid.csv,",
      "c03_judge_change_valid.csv, c04_county_residuals_valid.csv, c05_*_valid.csv, c05_missing_by_race.csv. No 2024 data used.")

C-02/C-03/C-04 and C-05 validation audit written: reports/c02_thresholds.csv, c03_group_metrics_valid.csv, c03_judge_change_valid.csv, c04_county_residuals_valid.csv, c05_*_valid.csv, c05_missing_by_race.csv. No 2024 data used.
